In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # 多模态脑MRI数据集探索
# 
# 这个notebook用于探索和验证多模态脑MRI数据集的结构完整性。

# ## 1. 导入必要的库

import os
from pathlib import Path
import pandas as pd
from datetime import datetime
import json

# ## 2. 设置数据路径和定义文件结构

# 根目录路径
ROOT_DIR = Path(r"W:\radiologie\mr-physik-data\Mitarbeiter\Jiayi\NEW_DATASET_ANALYSIS")

# 定义每个受试者应该包含的文件结构
REQUIRED_FILES = {
    "4D_image": "evaluated/realigned_coregistered/nibabel_stacked_normalized_skull_stripped.nii.gz",
    "3D_label": "seg/converted_alex_labels/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz"
}

# ## 3. 扫描和验证数据集

def scan_dataset(root_dir):
    """
    扫描根目录，找出所有符合条件的受试者文件夹并验证文件完整性
    
    Parameters:
    -----------
    root_dir : Path
        数据集根目录路径
    
    Returns:
    --------
    dict : 包含扫描结果的字典
    """
    
    results = {
        "scan_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "root_directory": str(root_dir),
        "total_subjects": 0,
        "valid_subjects": 0,
        "incomplete_subjects": 0,
        "subjects": []
    }
    
    # 检查根目录是否存在
    if not root_dir.exists():
        print(f"❌ 错误：根目录不存在 - {root_dir}")
        return results
    
    print(f"📁 扫描目录: {root_dir}\n")
    print("=" * 80)
    
    # 查找所有FOR_开头的文件夹
    subject_folders = [f for f in root_dir.iterdir() 
                      if f.is_dir() and f.name.startswith("FOR_")]
    
    results["total_subjects"] = len(subject_folders)
    
    if not subject_folders:
        print("⚠️ 警告：未找到任何以'FOR_'开头的文件夹")
        return results
    
    print(f"🔍 找到 {len(subject_folders)} 个受试者文件夹\n")
    
    # 检查每个受试者文件夹
    for idx, subject_folder in enumerate(subject_folders, 1):
        subject_info = {
            "subject_id": subject_folder.name,
            "path": str(subject_folder),
            "status": "完整",
            "missing_files": [],
            "existing_files": {}
        }
        
        print(f"[{idx}/{len(subject_folders)}] 检查受试者: {subject_folder.name}")
        
        # 检查必需的文件
        all_files_exist = True
        for file_type, relative_path in REQUIRED_FILES.items():
            file_path = subject_folder / relative_path
            
            if file_path.exists():
                subject_info["existing_files"][file_type] = str(file_path)
                print(f"  ✓ {file_type}: 找到")
                
                # 获取文件大小
                file_size_mb = file_path.stat().st_size / (1024 * 1024)
                subject_info["existing_files"][f"{file_type}_size_mb"] = round(file_size_mb, 2)
                
            else:
                all_files_exist = False
                subject_info["missing_files"].append(file_type)
                subject_info["status"] = "不完整"
                print(f"  ✗ {file_type}: 缺失")
        
        if all_files_exist:
            results["valid_subjects"] += 1
            print(f"  状态: ✅ 完整\n")
        else:
            results["incomplete_subjects"] += 1
            print(f"  状态: ⚠️ 不完整 - 缺失 {len(subject_info['missing_files'])} 个文件\n")
        
        results["subjects"].append(subject_info)
    
    return results

# 执行扫描
print("🚀 开始扫描数据集...\n")
scan_results = scan_dataset(ROOT_DIR)

# ## 4. 显示汇总信息

print("\n" + "=" * 80)
print("📊 数据集汇总信息")
print("=" * 80)
print(f"扫描时间: {scan_results['scan_time']}")
print(f"根目录: {scan_results['root_directory']}")
print(f"\n📈 统计信息:")
print(f"  • 总受试者数: {scan_results['total_subjects']}")
print(f"  • 完整数据受试者数: {scan_results['valid_subjects']} ({scan_results['valid_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")
print(f"  • 不完整数据受试者数: {scan_results['incomplete_subjects']} ({scan_results['incomplete_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")

# ## 5. 创建详细的数据框

# 创建受试者信息数据框
subjects_data = []
for subject in scan_results["subjects"]:
    row = {
        "受试者ID": subject["subject_id"],
        "状态": subject["status"],
        "4D影像": "✓" if "4D_image" in subject["existing_files"] else "✗",
        "3D标签": "✓" if "3D_label" in subject["existing_files"] else "✗",
        "缺失文件数": len(subject["missing_files"])
    }
    
    # 添加文件大小信息（如果存在）
    if "4D_image_size_mb" in subject["existing_files"]:
        row["4D影像大小(MB)"] = subject["existing_files"]["4D_image_size_mb"]
    if "3D_label_size_mb" in subject["existing_files"]:
        row["3D标签大小(MB)"] = subject["existing_files"]["3D_label_size_mb"]
    
    subjects_data.append(row)

df_subjects = pd.DataFrame(subjects_data)

print("\n📋 受试者详细信息:")
print(df_subjects.to_string(index=False))

# ## 6. 提取有效受试者路径列表

valid_subject_paths = []
valid_subject_info = []

for subject in scan_results["subjects"]:
    if subject["status"] == "完整":
        valid_subject_paths.append(subject["path"])
        valid_subject_info.append({
            "subject_id": subject["subject_id"],
            "path": subject["path"],
            "4d_image_path": subject["existing_files"]["4D_image"],
            "3d_label_path": subject["existing_files"]["3D_label"]
        })

print(f"\n✅ 找到 {len(valid_subject_paths)} 个数据完整的受试者")
print("\n有效受试者路径列表:")
for i, path in enumerate(valid_subject_paths, 1):
    print(f"{i}. {path}")

# ## 7. 保存结果

# 保存扫描结果为JSON文件
output_dir = Path("./mri_dataset_analysis_results")
output_dir.mkdir(exist_ok=True)

# 保存完整扫描结果
scan_results_file = output_dir / f"scan_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(scan_results_file, 'w', encoding='utf-8') as f:
    json.dump(scan_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 完整扫描结果已保存到: {scan_results_file}")

# 保存有效受试者信息
valid_subjects_file = output_dir / "valid_subjects.json"
with open(valid_subjects_file, 'w', encoding='utf-8') as f:
    json.dump(valid_subject_info, f, ensure_ascii=False, indent=2)

print(f"💾 有效受试者信息已保存到: {valid_subjects_file}")

# 保存为CSV格式（便于在Excel中查看）
csv_file = output_dir / f"subjects_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_subjects.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"💾 受试者汇总表已保存到: {csv_file}")

# ## 8. 数据质量检查

print("\n" + "=" * 80)
print("🔍 数据质量检查")
print("=" * 80)

# 检查不完整的受试者
incomplete_subjects = [s for s in scan_results["subjects"] if s["status"] == "不完整"]

if incomplete_subjects:
    print(f"\n⚠️ 发现 {len(incomplete_subjects)} 个数据不完整的受试者:")
    for subject in incomplete_subjects:
        print(f"\n受试者: {subject['subject_id']}")
        print(f"缺失文件: {', '.join(subject['missing_files'])}")
else:
    print("\n✅ 所有受试者数据完整！")

# 文件大小统计（仅针对完整数据）
if valid_subject_info:
    print("\n📊 文件大小统计（仅完整数据）:")
    
    sizes_4d = [s["existing_files"].get("4D_image_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    sizes_3d = [s["existing_files"].get("3D_label_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    
    if sizes_4d:
        print(f"\n4D影像文件:")
        print(f"  • 平均大小: {sum(sizes_4d)/len(sizes_4d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_4d):.2f} / {max(sizes_4d):.2f} MB")
    
    if sizes_3d:
        print(f"\n3D标签文件:")
        print(f"  • 平均大小: {sum(sizes_3d)/len(sizes_3d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_3d):.2f} / {max(sizes_3d):.2f} MB")

# ## 9. 快速访问有效数据的辅助函数

def get_subject_files(subject_id, valid_subjects=valid_subject_info):
    """
    根据受试者ID获取其文件路径
    
    Parameters:
    -----------
    subject_id : str
        受试者ID
    valid_subjects : list
        有效受试者信息列表
    
    Returns:
    --------
    dict : 包含文件路径的字典，如果未找到则返回None
    """
    for subject in valid_subjects:
        if subject["subject_id"] == subject_id:
            return {
                "4d_image": Path(subject["4d_image_path"]),
                "3d_label": Path(subject["3d_label_path"])
            }
    return None

# 示例：如何使用这个函数
if valid_subject_info:
    example_subject = valid_subject_info[0]["subject_id"]
    files = get_subject_files(example_subject)
    print(f"\n📌 示例：获取受试者 '{example_subject}' 的文件路径:")
    if files:
        print(f"  • 4D影像: {files['4d_image']}")
        print(f"  • 3D标签: {files['3d_label']}")

print("\n✨ 数据集探索完成！")
print(f"📁 所有结果已保存到: {output_dir.absolute()}")

# 将有效受试者路径列表存储为变量，便于后续使用
print(f"\n💡 提示：变量 'valid_subject_paths' 包含了所有数据完整的受试者路径")
print(f"        变量 'valid_subject_info' 包含了详细的文件路径信息")